In [27]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [28]:
# Constants
TRAIN_SIZE = 0.2
RANDOM_STATE = 1082

In [29]:
real_data_path = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
base_dir = "../../../Datasets/excel/2dSpoofingDataset/"
synthetic_files = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
]

In [30]:
# Load synthetic datasets
synthetic_dfs = {}
for file_name in synthetic_files:
    file_path = os.path.join(base_dir, file_name)
    try:
        synthetic_dfs[file_name] = pd.read_csv(file_path)
        print(f"Loaded {file_name}")
    except Exception as e:
        print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_excel(real_data_path)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

Loaded spoofing-CTGAN-synthetic.csv
Loaded spoofing-GaussianCopula-synthetic.csv
Loaded spoofing-TVAE-synthetic.csv
Loaded real data


In [31]:
# Preprocess real data
if real_df is not None:
    if 'Output' not in real_df.columns:
        raise ValueError("Real data does not have 'Output' column.")
    X_real = real_df.drop('Output', axis=1)
    y_real = real_df['Output']
    if y_real.dtype == object or y_real.dtype.name == 'category':
        le_real = LabelEncoder()
        y_real = le_real.fit_transform(y_real)
else:
    raise RuntimeError("Real data could not be loaded.")

In [32]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=5000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mae',
        random_state=RANDOM_STATE,
    )
}

In [33]:
for synth_name, synth_df in synthetic_dfs.items():
    print(f"Train on synthetic: {synth_name}, testing on real data")

    if 'Output' not in synth_df.columns:
        print(f"Skipping {synth_name}: 'Output' column not found.")
        continue

    X_synth = synth_df.drop('Output', axis=1)
    y_synth = synth_df['Output']
    if y_synth.dtype == object or y_synth.dtype.name == 'category':
        le_synth = LabelEncoder()
        y_synth = le_synth.fit_transform(y_synth)

    X_synth_train, _, y_synth_train, _ = train_test_split(
        X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth
    )

    # Scale based on synthetic training split, apply to both train and real test
    scaler = StandardScaler()
    X_synth_train_scaled = scaler.fit_transform(X_synth_train)
    X_real_scaled = scaler.transform(X_real)

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("Training on synthetic")
        model.fit(X_synth_train_scaled, y_synth_train)
        print("Evaluating on real data...")
        y_pred = model.predict(X_real_scaled)
        acc = accuracy_score(y_real, y_pred)
        print(f"Accuracy: {acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_real, y_pred, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_real, y_pred))

Train on synthetic: spoofing-CTGAN-synthetic.csv, testing on real data

Logistic Regression
Training on synthetic
Evaluating on real data...
Accuracy: 0.4764

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.50      0.64    397825
           1       0.18      0.39      0.25     36458
           2       0.18      0.26      0.21     44232
           3       0.13      0.52      0.20     32015

    accuracy                           0.48    510530
   macro avg       0.34      0.42      0.33    510530
weighted avg       0.71      0.48      0.54    510530

Confusion Matrix:
[[200432  53571  44392  99430]
 [ 11066  14366   4258   6768]
 [ 13163   8483  11657  10929]
 [  8690   2200   4386  16739]]

MLP Classifier
Training on synthetic
Evaluating on real data...
Accuracy: 0.5254

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.45      0.62    397825
           1       0.31  